# Drop All Schemes and Tables

This notebook drops all tables in the listed created using the definition in fabric/notebooks/schema folder 
- Skips schemes and tables that do not exist.
- Prints status for each drop operation.

In [ ]:
# First, let's see what schemas and tables actually exist
print("=== CURRENT SCHEMAS ===")
try:
    schemas = spark.sql("SHOW DATABASES").collect()
    for schema in schemas:
        schema_name = schema[0]
        print(f"\n📁 Schema: {schema_name}")
        try:
            tables = spark.sql(f"SHOW TABLES IN {schema_name}").collect()
            if tables:
                for table in tables:
                    print(f"  📋 {table.tableName}")
            else:
                print("  (no tables)")
        except Exception as e:
            print(f"  ⚠️ Error accessing tables: {e}")
except Exception as e:
    print(f"⚠️ Error getting schemas: {e}")

In [ ]:
# Dynamic schema and table discovery - NO HARDCODING!
def get_user_schemas():
    """Get all schemas, excluding system ones"""
    try:
        all_schemas = spark.sql("SHOW DATABASES").collect()
        # Filter out system schemas (add more system schema names as needed)
        system_schemas = {'default', 'information_schema', 'sys', 'hive_metastore'}
        user_schemas = [schema[0] for schema in all_schemas if schema[0] not in system_schemas]
        return user_schemas
    except Exception as e:
        print(f"⚠️ Error getting schemas: {e}")
        return []

def get_schema_tables(schema_name):
    """Get all tables in a schema"""
    try:
        tables = spark.sql(f"SHOW TABLES IN {schema_name}").collect()
        return [table.tableName for table in tables]
    except Exception as e:
        print(f"⚠️ Error getting tables for {schema_name}: {e}")
        return []

def drop_all_user_data(exclude_schemas=None, exclude_tables=None):
    """
    Drop all user schemas and tables dynamically
    
    Args:
        exclude_schemas: List of schema names to skip (optional)
        exclude_tables: Dict of {schema: [table_list]} to skip specific tables (optional)
    """
    exclude_schemas = exclude_schemas or []
    exclude_tables = exclude_tables or {}
    
    print("🔍 Discovering all user schemas and tables...")
    user_schemas = get_user_schemas()
    
    if not user_schemas:
        print("✅ No user schemas found!")
        return
    
    print(f"📁 Found schemas: {', '.join(user_schemas)}")
    
    # Step 1: Drop all tables in each schema
    for schema in user_schemas:
        if schema in exclude_schemas:
            print(f"⏭️ Skipping excluded schema: {schema}")
            continue
            
        print(f"\n--- Processing schema: {schema} ---")
        tables = get_schema_tables(schema)
        
        if not tables:
            print(f"  📭 No tables in {schema}")
            continue
            
        tables_to_skip = exclude_tables.get(schema, [])
        tables_to_drop = [t for t in tables if t not in tables_to_skip]
        
        if tables_to_skip:
            print(f"  ⏭️ Skipping tables: {', '.join(tables_to_skip)}")
        
        for table in tables_to_drop:
            full_table = f"{schema}.{table}"
            print(f"  🗑️ Dropping {full_table}...")
            try:
                spark.sql(f"DROP TABLE IF EXISTS {full_table}")
                print(f"  ✅ {full_table} dropped!")
            except Exception as e:
                print(f"  ⚠️ Could not drop {full_table}: {e}")
    
    # Step 2: Drop empty schemas
    print(f"\n--- Dropping empty schemas ---")
    for schema in user_schemas:
        if schema in exclude_schemas:
            continue
            
        try:
            remaining_tables = get_schema_tables(schema)
            if remaining_tables:
                print(f"⚠️ Cannot drop {schema}: Still has tables: {remaining_tables}")
            else:
                print(f"🗑️ Dropping empty schema: {schema}")
                spark.sql(f"DROP DATABASE IF EXISTS {schema} CASCADE")
                print(f"✅ Schema {schema} dropped!")
        except Exception as e:
            print(f"⚠️ Could not drop schema {schema}: {e}")

# === USAGE OPTIONS ===

# Option 1: Drop EVERYTHING (most common use case)
print("🚨 DROPPING ALL USER SCHEMAS AND TABLES")
drop_all_user_data()

# Option 2: Drop everything except specific schemas (uncomment to use)
# print("🚨 DROPPING ALL EXCEPT EXCLUDED SCHEMAS")
# drop_all_user_data(exclude_schemas=['keep_this_schema', 'and_this_one'])

# Option 3: Drop everything except specific tables (uncomment to use)  
# print("🚨 DROPPING ALL EXCEPT EXCLUDED TABLES")
# drop_all_user_data(exclude_tables={
#     'shared': ['important_customer_table'],
#     'finance': ['critical_payment_data']
# })

print(f"\n🎉 CLEANUP COMPLETE!")